# AID2026 — Incident Detection Pipeline
**Team Submission** | Google Colab Pro+ (A100)

This notebook covers:
1. Environment setup
2. Dataset preparation
3. Model training
4. Evaluation
5. Official test submission (`test.py`)

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q transformers==4.40.0 accelerate decord torchvision torchaudio
!pip install -q scikit-learn opencv-python-headless einops timm
print('Dependencies installed.')

In [ ]:
# Mount Google Drive (where dataset lives)
from google.colab import drive
drive.mount('/content/drive')

# Set your dataset path here
DATASET_PATH = '/content/drive/MyDrive/AID2026/dataset'  # ← EDIT THIS
ANNOTATIONS_CSV = f'{DATASET_PATH}/annotations.csv'

import os
print('Files:', os.listdir(DATASET_PATH)[:10])

In [ ]:
# Clone / copy project files
import shutil, os

PROJECT_DIR = '/content/aid2026'
os.makedirs(PROJECT_DIR, exist_ok=True)

# If uploaded as zip:
# !unzip /content/drive/MyDrive/AID2026/aid2026.zip -d /content/

# Or copy from Drive
# shutil.copytree('/content/drive/MyDrive/AID2026/aid2026', PROJECT_DIR, dirs_exist_ok=True)

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

## 2. Dataset Preparation

In [ ]:
VIDEOS_DIR = f'{DATASET_PATH}/videos'
DATA_DIR   = f'{PROJECT_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

!python scripts/prepare_data.py \
    --csv {ANNOTATIONS_CSV} \
    --output_dir {DATA_DIR} \
    --videos_dir {VIDEOS_DIR} \
    --val_ratio 0.15 \
    --seed 42

In [ ]:
import json
with open(f'{DATA_DIR}/split_info.json') as f:
    split_info = json.load(f)
print(json.dumps(split_info, indent=2))

## 3. Training

In [ ]:
# Training configuration
import json

cfg = {
    # Model
    'backbone':       'MCG-NJU/videomae-small',
    'clip_frames':    16,
    'context_clips':  8,
    'dropout':        0.3,
    'freeze_layers':  8,
    # Data
    'train_csv':      f'{DATA_DIR}/train.csv',
    'val_csv':        f'{DATA_DIR}/val.csv',
    'videos_dir':     VIDEOS_DIR,
    'clip_hop':       8,
    'max_clips':      48,
    # Training
    'batch_size':     4,
    'num_workers':    2,
    'epochs':         30,
    'lr':             2e-4,
    'weight_decay':   1e-4,
    'patience':       8,
    'amp':            True,
    'checkpoint_dir': f'{PROJECT_DIR}/checkpoints',
}

cfg_path = f'{PROJECT_DIR}/configs/train_config.json'
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print('Config saved:', cfg_path)

In [ ]:
# Run training
!python src/train.py --config {cfg_path}

## 4. Evaluation & Metrics Visualization

In [ ]:
import json
import matplotlib.pyplot as plt

log_path = f'{PROJECT_DIR}/checkpoints/metrics_log.jsonl'
epochs, f1s, precs, recs, delays = [], [], [], [], []

with open(log_path) as f:
    for line in f:
        m = json.loads(line)
        epochs.append(m['epoch'])
        f1s.append(m['f1'])
        precs.append(m['precision'])
        recs.append(m['recall'])
        delays.append(m['avg_delay_sec'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, f1s,    label='F1',        linewidth=2)
axes[0].plot(epochs, precs,  label='Precision',  linewidth=2, linestyle='--')
axes[0].plot(epochs, recs,   label='Recall',     linewidth=2, linestyle=':')
axes[0].set_title('Validation Metrics'); axes[0].legend(); axes[0].grid(True)
axes[0].axhline(y=max(f1s), color='green', linewidth=1, alpha=0.5)
axes[0].text(epochs[-1], max(f1s)+0.01, f'Best F1={max(f1s):.4f}', ha='right', color='green')

axes[1].plot(epochs, delays, color='orange', linewidth=2)
axes[1].set_title('Avg Notification Delay (s)'); axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best F1: {max(f1s):.4f} at epoch {epochs[f1s.index(max(f1s))]}')

## 5. Official Test Submission

In [ ]:
# Point this to the contest test set
TEST_VIDEOS_DIR = '/content/drive/MyDrive/AID2026/test_videos'  # ← EDIT
RESULTS_DIR     = f'{PROJECT_DIR}/results'

!python test.py --videos {TEST_VIDEOS_DIR} --results {RESULTS_DIR}

# Preview results
import pandas as pd
df = pd.read_csv(f'{RESULTS_DIR}/results.csv')
print(f'Total predictions: {len(df)}')
print(f'Incidents detected: {df["start"].notna().sum()}')
df.head(20)

In [ ]:
# Package submission archive
import shutil

SUBMISSION_DIR = '/content/submission'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Copy required files
shutil.copy(f'{PROJECT_DIR}/test.py',                          f'{SUBMISSION_DIR}/test.py')
shutil.copytree(f'{PROJECT_DIR}/src',                          f'{SUBMISSION_DIR}/src',      dirs_exist_ok=True)
shutil.copytree(f'{PROJECT_DIR}/checkpoints',                  f'{SUBMISSION_DIR}/checkpoints', dirs_exist_ok=True)
shutil.copy('/content/aid2026.ipynb',                          f'{SUBMISSION_DIR}/test.ipynb')

# Create zip
shutil.make_archive('/content/drive/MyDrive/AID2026/submission', 'zip', SUBMISSION_DIR)
print('Submission archive created!')
print('Files:', os.listdir(SUBMISSION_DIR))